### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
cd Your_Dir/emg2qwerty

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [ ]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  # --multirun

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

In [ ]:
import re
from pathlib import Path

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib.pyplot as plt


# Extract train loss from log file with regex
log_file = "manual_log4.txt"  

# Go line by line to extract loss values for each epoch
train_losses = {}
with open(log_file) as f:
    for line in f:
        m = re.search(r"Epoch (\d+): 100%.*?loss=([\d.]+)", line)
        if m:
            epoch = int(m.group(1))
            if epoch not in train_losses:
                train_losses[epoch] = float(m.group(2))

train_epochs = sorted(train_losses.keys())
train_vals = [train_losses[e] for e in train_epochs]


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from tbparse import SummaryReader

# --- Research paper style ---
matplotlib.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})

reader = SummaryReader("logs/2026-03-13/10-08-59/lightning_logs/version_0")
df = reader.scalars

epoch_map = df[df["tag"] == "epoch"].drop_duplicates(subset="step", keep="last").set_index("step")["value"]

TRAIN_COLOR = "#7b2d8e"
VAL_COLOR = "#e8870e"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))

# --- (a) Loss vs Epoch ---
ax1.plot(list(range(0, 300)), train_vals, label = "Train", color=TRAIN_COLOR)

for tag, label, color in [
    ("train/loss_epoch", "Train", TRAIN_COLOR),
    ("val/loss", "Validation", VAL_COLOR),
]:
    sub = df[df["tag"] == tag].copy()
    if not sub.empty:
        sub["epoch"] = sub["step"].map(epoch_map).ffill()
        grouped = sub.groupby("epoch")["value"].mean()
        ax1.plot(grouped.index, grouped.values, label=label, color=color)
ax1.set_ylim(0, 4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("(a) Loss", style="italic")
ax1.legend(frameon=True, fancybox=False, edgecolor="0.7")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.tick_params(direction="in")

# --- (b) CER vs Epoch ---
for tag, label, color in [
    ("train/CER", "Train", TRAIN_COLOR),
    ("val/CER", "Validation", VAL_COLOR),
]:
    sub = df[df["tag"] == tag].copy()
    if not sub.empty:
        sub["epoch"] = sub["step"].map(epoch_map).ffill()
        grouped = sub.groupby("epoch")["value"].mean()
        ax2.plot(grouped.index, grouped.values, label=label, color=color)
ax2.set_ylim(0, 125)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("CER")
ax2.set_title("(b) Character Error Rate", style="italic")
ax2.legend(frameon=True, fancybox=False, edgecolor="0.7")
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
ax2.tick_params(direction="in")
ax1.set_xlim(0, 150)
ax2.set_xlim(0, 150)
plt.tight_layout(w_pad=3.0)
plt.savefig("train_Transformer.png")
# plt.savefig("train_Transformer.pdf")
plt.show()